In [ ]:
# Préparation commune des TP Python
# Le notebook utilise uniquement les ressources fournies dans le dossier codes/.
from pathlib import Path
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib'}
missing = [package for module, package in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installation des paquets manquants :", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

WORKDIR = Path.cwd().resolve()
CODES_DIR = WORKDIR.parent if WORKDIR.name == "correction" else WORKDIR
TOOLBOX_DIR = CODES_DIR / "toolbox"
DATA_DIR = CODES_DIR / "data"
if not CODES_DIR.is_dir():
    raise FileNotFoundError("Exécutez le notebook depuis le dossier codes/." )
if TOOLBOX_DIR.is_dir():
    sys.path.insert(0, str(TOOLBOX_DIR))

DICE_FILE = TOOLBOX_DIR / "DICE.py"
if not DICE_FILE.is_file():
    raise FileNotFoundError(
        f"Module du cours introuvable : {DICE_FILE}. "
        "Téléchargez le dossier codes complet, avec toolbox/."
    )

print("Environnement prêt :", CODES_DIR)


# TP2 — Modèle climatique simplifié et sensibilité

**Date de la séance :** mercredi 2 septembre 2026, 10:45–12:45

**Objectifs**
- Prendre en main le modèle climat-économie DICE
- Simuler une trajectoire de référence 2015-2100
- Étudier l'effet de la sensibilité climatique (T2XCO2) sur la température
- Comprendre l'inertie du système climatique

**Prérequis** : TP1, notions de base sur les modèles climatiques simplifiés (bloc carbone / bloc température).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import DICE

## Partie 1 — Prise en main du modèle DICE

**Question 1.** Importez DICE, créez les paramètres par défaut et lancez une simulation de référence.

*Note technique* : `DICE.update_path` s'utilise avec la signature `update_path(sim, timevec, p)`, où `timevec` est la liste des indices de temps à simuler (typiquement `range(1, p.nT)`, car l'indice 0 correspond à l'état initial 2015).

In [ ]:
p = DICE.Params()
path = DICE.init_states(p)
timevec = range(1, p.nT)
path = DICE.update_path(path, timevec, p)
print(f"Simulation de {p.t0} à {p.tT} — {p.nT} périodes de {p.Delta} ans")

**Question 2.** Tracez les trajectoires de T_AT (température), E (émissions), M_AT (CO2 atmosphérique) de 2015 à 2100.

In [ ]:
# Expected output: T_AT croît d'environ 0.85°C (2015) à ~3-4°C (2100) dans ce scénario sans politique climatique
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(path[:, p.i_time], path[:, p.i_T_AT], color='tab:red')
axes[0].set_title('Température atmosphérique T_AT (°C)')
axes[0].set_xlabel('Année'); axes[0].grid(True)

axes[1].plot(path[:, p.i_time], path[:, p.i_E], color='tab:orange')
axes[1].set_title('Émissions de CO2 E (GtCO2/période)')
axes[1].set_xlabel('Année'); axes[1].grid(True)

axes[2].plot(path[:, p.i_time], path[:, p.i_M_AT], color='tab:brown')
axes[2].set_title('Carbone atmosphérique M_AT (GtC)')
axes[2].set_xlabel('Année'); axes[2].grid(True)
plt.tight_layout()
plt.show()

## Partie 2 — Sensibilité climatique

**Question 3.** La sensibilité climatique T2XCO2 mesure le réchauffement à l'équilibre pour un doublement du CO2. Simulez le modèle avec T2XCO2 = 1.5, 3.1 (référence), 4.5. Tracez les trajectoires de T_AT pour les trois cas.

In [ ]:
sensitivities = [1.5, 3.1, 4.5]
labels = ['T2XCO2 = 1.5°C (faible)', 'T2XCO2 = 3.1°C (référence)', 'T2XCO2 = 4.5°C (élevée)']
paths = []
for T2X in sensitivities:
    p_s = DICE.Params(T2XCO2=T2X)
    path_s = DICE.init_states(p_s)
    timevec_s = range(1, p_s.nT)
    path_s = DICE.update_path(path_s, timevec_s, p_s)
    paths.append(path_s)

In [ ]:
# Expected output: écart net en 2100 entre les trois courbes, l'écart s'accroît avec l'horizon temporel
plt.figure(figsize=(9, 5))
for path_s, label in zip(paths, labels):
    plt.plot(path_s[:, p.i_time], path_s[:, p.i_T_AT], label=label, linewidth=2)
plt.xlabel('Année')
plt.ylabel('T_AT (°C)')
plt.title("Sensibilité climatique et trajectoire de température")
plt.legend()
plt.grid(True)
plt.show()

**Question 4.** Calculez l'écart de température en 2100 entre le cas le plus favorable et le plus défavorable. Que représente cet écart pour la gestion des risques ?

In [ ]:
# Expected output: écart de l'ordre de 2 à 3°C en 2100 entre T2XCO2=1.5 et T2XCO2=4.5
T_2100_low = paths[0][-1, p.i_T_AT]
T_2100_high = paths[-1][-1, p.i_T_AT]
print(f"T_AT en 2100, sensibilité faible (1.5) : {T_2100_low:.2f}°C")
print(f"T_AT en 2100, sensibilité élevée (4.5) : {T_2100_high:.2f}°C")
print(f"Écart : {T_2100_high - T_2100_low:.2f}°C")

## Partie 3 — Inertie climatique

**Question 5.** Simulez un scénario où les émissions tombent à 0 en 2030 (mu=1 à partir de 2030). La température continue-t-elle de monter ? Pourquoi ?

In [ ]:
# Expected output: T_AT continue à croître légèrement après 2030 malgré des émissions nettes nulles,
# du fait de l'inertie thermique de l'océan (T_LO qui rattrape T_AT) et du stock de CO2 déjà accumulé
p_zero = DICE.Params()
path_zero = DICE.init_states(p_zero)
# 2030 correspond à l'indice t tel que time = 2030 -> t = (2030-2015)/5 = 3
year_col = p_zero.t0 + p_zero.Delta * (np.arange(1, p_zero.nT))
idx_2030 = np.where(year_col >= 2030)[0][0] + 1
path_zero[1:idx_2030, p_zero.i_mu] = 0.03
path_zero[idx_2030:, p_zero.i_mu] = 1.0
timevec_zero = range(1, p_zero.nT)
path_zero = DICE.update_path(path_zero, timevec_zero, p_zero)

plt.figure(figsize=(9, 5))
plt.plot(path_zero[:, p_zero.i_time], path_zero[:, p_zero.i_T_AT], color='tab:red', label='T_AT')
plt.axvline(2030, color='grey', linestyle='--', label='Arrêt des émissions nettes')
plt.xlabel('Année'); plt.ylabel('T_AT (°C)')
plt.title("Inertie climatique : la température continue de monter après l'arrêt des émissions")
plt.legend(); plt.grid(True)
plt.show()

## Interprétation

Répondez en quelques phrases :
1. Quelle est la température atmosphérique prévue en 2100 dans le scénario de référence ?
2. Pourquoi la sensibilité climatique est-elle une source d'incertitude majeure ?
3. Qu'est-ce que l'inertie climatique et quelles sont ses implications pour les risques à long terme ?
4. Dans quel sens cette incertitude affecte-t-elle la tarification d'un produit d'assurance catastrophe ?

### Éléments de réponse

1. Dans le scénario de référence (T2XCO2=3.1, pas de politique climatique), T_AT atteint environ 3 à 4°C en 2100.
2. La sensibilité climatique n'est pas connue avec précision : le GIEC AR6 l'estime dans une fourchette probable de 2,5 à 4°C (intervalle *likely*), avec des scénarios extrêmes crédibles en dehors de cette plage. Cette incertitude irréductible se traduit directement en incertitude sur les températures futures pour un même scénario d'émissions.
3. L'inertie climatique désigne le décalage entre la stabilisation des émissions/concentrations et la stabilisation de la température, dû à la capacité thermique de l'océan profond. Cela implique qu'un stock de réchauffement est déjà « engagé » même après l'arrêt des émissions, ce qui a des conséquences directes sur l'horizon d'exposition aux risques physiques pour un assureur.
4. Cette incertitude doit conduire à intégrer une marge de prudence dans la tarification et à ne pas se limiter à un scénario central : il est nécessaire de raisonner en distribution de scénarios (cf. TP5) plutôt qu'en projection déterministe unique.